## 1. Imports and Configuration

In [1]:
import warnings
from pathlib import Path

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from mlflow.tracking import MlflowClient
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
EXPERIMENT_NAME = "anomaly-detection"
REGISTERED_MODEL_NAME = "anomaly-detector-xgb-smote"
PRODUCTION_MODEL_NAME = "anomaly-detection-prod"
TARGET_COL = "Class"
IMAGE_SIZE = (32, 32)
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp"}
CLASS_FOLDERS = {"no": 0, "yes": 1}

# Paths relative to Capstone Project root
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "data" / "dataset"
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"


def load_image_dataset(data_dir: Path) -> pd.DataFrame:
    """Convert yes/no image folders into a tabular feature matrix."""
    feature_rows = []
    labels = []
    file_names = []

    for folder_name, label in CLASS_FOLDERS.items():
        class_dir = data_dir / folder_name
        if not class_dir.is_dir():
            raise FileNotFoundError(f"Missing class folder: {class_dir}")

        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
                continue
            img = Image.open(img_path).convert("L")
            img = img.resize(IMAGE_SIZE)
            pixels = np.array(img, dtype=np.float32).flatten() / 255.0
            feature_rows.append(pixels)
            labels.append(label)
            file_names.append(img_path.name)

    if not feature_rows:
        raise FileNotFoundError(
            f"No images found in {data_dir}/no or {data_dir}/yes. "
            "Add .png/.jpg files to both folders."
        )

    n_features = feature_rows[0].shape[0]
    feature_cols = [f"pixel_{i}" for i in range(n_features)]
    features_df = pd.DataFrame(feature_rows, columns=feature_cols)
    features_df[TARGET_COL] = labels
    features_df["filename"] = file_names
    return features_df


mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Project root: {BASE_DIR}")
print(f"Dataset folder: {DATA_DIR}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

C:\Users\hp\Downloads\Capstone Project\venv\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\hp\Downloads\Capstone Project\venv\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
2026/06/21 20:54:05 INFO mlflow.tracking.fluent: Experiment with name 'anomaly-detection' does not exist. Creating a new experiment.


Project root: C:\Users\hp\Downloads\Capstone Project
Dataset folder: C:\Users\hp\Downloads\Capstone Project\data\dataset
MLflow tracking URI: http://127.0.0.1:5000
Experiment: anomaly-detection


## 2. Load Dataset and Inspect Class Distribution

Images from `data/dataset/no/` (class 0) and `data/dataset/yes/` (class 1) are converted to numerical pixel features for the required scikit-learn / XGBoost models.

In [2]:
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset folder not found at {DATA_DIR}. "
        "Create data/dataset/no/ and data/dataset/yes/ and add images."
    )

df = load_image_dataset(DATA_DIR)
print(f"Shape: {df.shape}")
print(f"Feature columns: {df.shape[1] - 2} pixel features + Class + filename")
print(f"Images loaded — no: {(df[TARGET_COL] == 0).sum()} | yes: {(df[TARGET_COL] == 1).sum()}")
df[[TARGET_COL, "filename"]].head()

Shape: (193, 1026)
Feature columns: 1024 pixel features + Class + filename
Images loaded — no: 89 | yes: 104


,Class,filename
0,0,AdobeStock_620725543_Preview.png
1,0,AdobeStock_661747653_Preview.png
2,0,AdobeStock_676343536_Preview.png
3,0,AdobeStock_680993600_Preview.png
4,0,AdobeStock_683749437_Preview.png


In [3]:
class_counts = df[TARGET_COL].value_counts().sort_index()
class_ratio = df[TARGET_COL].value_counts(normalize=True).sort_index()

print("Class counts (0 = no, 1 = yes):")
print(class_counts)
print("\nClass proportions:")
print(class_ratio)

majority_pct = class_ratio.max() * 100
minority_pct = class_ratio.min() * 100
print(f"\nImbalance ratio: {majority_pct:.2f}% : {minority_pct:.2f}%")
print(f"(Assignment requires at least 80:20 — this dataset is ~{majority_pct:.0f}:{minority_pct:.0f})")

Class counts (0 = no, 1 = yes):
Class
0     89
1    104
Name: count, dtype: int64

Class proportions:
Class
0    0.46114
1    0.53886
Name: proportion, dtype: float64

Imbalance ratio: 53.89% : 46.11%
(Assignment requires at least 80:20 — this dataset is ~54:46)


## 3. Stratified Train-Test Split (70% / 30%)

The **test set stays untouched** until final evaluation in each experiment.

In [4]:
X = df.drop([TARGET_COL, "filename"], axis=1)
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

Train size: 135 | Test size: 58

Train class distribution:
Class
1    0.540741
0    0.459259
Name: proportion, dtype: float64

Test class distribution:
Class
1    0.534483
0    0.465517
Name: proportion, dtype: float64


## 4. SMOTE on Training Set Only

In [5]:
print("Before SMOTE (training set only):")
print(y_train.value_counts())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE (training set only):")
print(pd.Series(y_train_smote).value_counts())

print(f"\nTraining samples before SMOTE: {len(X_train)}")
print(f"Training samples after SMOTE:  {len(X_train_smote)}")

Before SMOTE (training set only):
Class
1    73
0    62
Name: count, dtype: int64

After SMOTE (training set only):
Class
0    73
1    73
Name: count, dtype: int64

Training samples before SMOTE: 135
Training samples after SMOTE:  146


## 5. Metric Helper Function

In [6]:
def get_metrics(y_true, y_pred):
    """Extract the four required metrics from classification_report."""
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "accuracy": report["accuracy"],
        "recall_class_0": report["0"]["recall"],
        "recall_class_1": report["1"]["recall"],
        "f1_score_macro": report["macro avg"]["f1-score"],
    }


def log_metrics_to_mlflow(metrics: dict) -> None:
    for name, value in metrics.items():
        mlflow.log_metric(name, value)

## 6. Experiment 1 — Logistic Regression (Baseline)

In [7]:
with mlflow.start_run(run_name="Logistic Regression"):
    lr_params = {"C": 1, "solver": "liblinear", "random_state": RANDOM_STATE}

    model = LogisticRegression(**lr_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("C", lr_params["C"])
    mlflow.log_param("solver", lr_params["solver"])
    mlflow.log_param("used_smote", False)
    log_metrics_to_mlflow(metrics)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print("Logistic Regression metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

2026/06/21 20:54:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logistic Regression metrics:
  accuracy: 0.4483
  recall_class_0: 0.3333
  recall_class_1: 0.5484
  f1_score_macro: 0.4376
🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/594281079765530894/runs/95deb719070c48ba88f634db26a2ce17
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/594281079765530894


## 7. Experiment 2 — Random Forest

In [8]:
with mlflow.start_run(run_name="Random Forest"):
    rf_params = {
        "n_estimators": 30,
        "max_depth": 3,
        "random_state": RANDOM_STATE,
    }

    model = RandomForestClassifier(**rf_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("n_estimators", rf_params["n_estimators"])
    mlflow.log_param("max_depth", rf_params["max_depth"])
    mlflow.log_param("used_smote", False)
    log_metrics_to_mlflow(metrics)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print("Random Forest metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

2026/06/21 20:54:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest metrics:
  accuracy: 0.6207
  recall_class_0: 0.4815
  recall_class_1: 0.7419
  f1_score_macro: 0.6091
🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/594281079765530894/runs/f41fbcf586a944bba6eb0bd47ec9eab0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/594281079765530894


## 8. Experiment 3 — XGBoost (Original Imbalanced Data)

In [9]:
with mlflow.start_run(run_name="XGBClassifier"):
    xgb_params = {
        "eval_metric": "logloss",
        "random_state": RANDOM_STATE,
    }

    model = XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("eval_metric", xgb_params["eval_metric"])
    mlflow.log_param("used_smote", False)
    log_metrics_to_mlflow(metrics)
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("XGBClassifier metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

2026/06/21 20:55:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBClassifier metrics:
  accuracy: 0.5690
  recall_class_0: 0.5185
  recall_class_1: 0.6129
  f1_score_macro: 0.5657
🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/594281079765530894/runs/beace459cd8e4e848b735ff4c096f286
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/594281079765530894


## 9. Experiment 4 — XGBoost with SMOTE

In [10]:
with mlflow.start_run(run_name="XGBClassifier With SMOTE"):
    xgb_params = {
        "eval_metric": "logloss",
        "random_state": RANDOM_STATE,
    }

    model = XGBClassifier(**xgb_params)
    model.fit(X_train_smote, y_train_smote)
    preds = model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    mlflow.log_param("eval_metric", xgb_params["eval_metric"])
    mlflow.log_param("used_smote", True)
    log_metrics_to_mlflow(metrics)
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("XGBClassifier With SMOTE metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

2026/06/21 20:55:32 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBClassifier With SMOTE metrics:
  accuracy: 0.5000
  recall_class_0: 0.4815
  recall_class_1: 0.5161
  f1_score_macro: 0.4987
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/594281079765530894/runs/54e6fc8373e94737a5164d71f9e6fb7a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/594281079765530894


## 10. Compare All Runs

In [11]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.recall_class_1 DESC", "metrics.f1_score_macro DESC"],
)

display_cols = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.recall_class_0",
    "metrics.recall_class_1",
    "metrics.f1_score_macro",
]
available_cols = [c for c in display_cols if c in runs_df.columns]
comparison = runs_df[available_cols].copy()
comparison

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.recall_class_0,metrics.recall_class_1,metrics.f1_score_macro
0,f41fbcf586a944bba6eb0bd47ec9eab0,Random Forest,0.620690,0.481481,0.741935,0.609069
1,beace459cd8e4e848b735ff4c096f286,XGBClassifier,0.568966,0.518519,0.612903,0.565738
2,95deb719070c48ba88f634db26a2ce17,Logistic Regression,0.448276,0.333333,0.548387,0.437576
3,54e6fc8373e94737a5164d71f9e6fb7a,XGBClassifier With SMOTE,0.500000,0.481481,0.516129,0.498659


## 11. Model Registry — Register Best Model

Best model is selected by **recall_class_1** first, then **f1_score_macro**.

In [12]:
best_run = runs_df.iloc[0]
best_run_id = best_run["run_id"]
best_run_name = best_run.get("tags.mlflow.runName", "unknown")

print(f"Best run: {best_run_name}")
print(f"Run ID: {best_run_id}")
print(f"recall_class_1: {best_run['metrics.recall_class_1']:.4f}")
print(f"f1_score_macro: {best_run['metrics.f1_score_macro']:.4f}")

model_uri = f"runs:/{best_run_id}/model"

Best run: Random Forest
Run ID: f41fbcf586a944bba6eb0bd47ec9eab0
recall_class_1: 0.7419
f1_score_macro: 0.6091


In [13]:
client = MlflowClient()

# Register best model
registered = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
challenger_version = registered.version
print(f"Registered {REGISTERED_MODEL_NAME} version {challenger_version}")

# Assign @challenger alias
client.set_registered_model_alias(
    REGISTERED_MODEL_NAME,
    "challenger",
    challenger_version,
)
print(f"Alias @challenger assigned to version {challenger_version}")

Successfully registered model 'anomaly-detector-xgb-smote'.
2026/06/21 20:55:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: anomaly-detector-xgb-smote, version 1
Created version '1' of model 'anomaly-detector-xgb-smote'.


Registered anomaly-detector-xgb-smote version 1
Alias @challenger assigned to version 1


## 12. Copy to Production Registry and Assign @champion

In [14]:
copied = client.copy_model_version(
    src_model_uri=f"models:/{REGISTERED_MODEL_NAME}/{challenger_version}",
    dst_name=PRODUCTION_MODEL_NAME,
)
prod_version = copied.version
print(f"Copied to {PRODUCTION_MODEL_NAME} version {prod_version}")

client.set_registered_model_alias(
    PRODUCTION_MODEL_NAME,
    "champion",
    prod_version,
)
print(f"Alias @champion assigned to {PRODUCTION_MODEL_NAME} version {prod_version}")

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'anomaly-detector-xgb-smote' to version '1' of model 'anomaly-detection-prod'.


Copied to anomaly-detection-prod version 1
Alias @champion assigned to anomaly-detection-prod version 1


## 13. Load Production Model and Run Inference on Test Set

In [15]:
prod_model_uri = f"models:/{PRODUCTION_MODEL_NAME}@champion"
prod_model = mlflow.pyfunc.load_model(prod_model_uri)

final_preds = prod_model.predict(X_test)
final_metrics = get_metrics(y_test, final_preds)

print(f"Production model loaded from: {prod_model_uri}")
print("\nFinal test-set metrics (production @champion):")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")

print("\nClassification report:")
print(classification_report(y_test, final_preds, zero_division=0))

Production model loaded from: models:/anomaly-detection-prod@champion

Final test-set metrics (production @champion):
  accuracy: 0.6207
  recall_class_0: 0.4815
  recall_class_1: 0.7419
  f1_score_macro: 0.6091

Classification report:
              precision    recall  f1-score   support

           0       0.62      0.48      0.54        27
           1       0.62      0.74      0.68        31

    accuracy                           0.62        58
   macro avg       0.62      0.61      0.61        58
weighted avg       0.62      0.62      0.61        58

